In [1]:
import data_helper
df_raw = data_helper.load('h',0,1,1,1)

In [2]:
df = df_raw.copy()
df['Total_Wind'] = df['Wind Offshore'] +  df['Wind Onshore'] 

In [ ]:
import numpy as np

df['Wind_Log'] = np.log1p(df['Total_Wind'])

noise_threshold = 0.1 

# If physical solar generation exceeds the noise floor, keep it. 
# Otherwise, force it to absolute zero.
df['Solar_Gated'] = np.where(df['Solar'] > noise_threshold, df['Solar'], 0.0)

df['Solar_Log'] = np.log1p(df['Solar_Gated'])

In [9]:
import pandas as pd
import matplotlib.pyplot as plt
from neuralforecast import NeuralForecast
from neuralforecast.models import XLinear
from neuralforecast.losses.pytorch import MAE
from sklearn.metrics import mean_absolute_error
import optuna

c:\Users\scben\Documents\Obsidian\Benedek\Project-Thesis\BEN\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-05-22 12:37:31,296	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.
2026-05-22 12:37:32,193	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


In [ ]:
df_uni = df.copy()

# ==========================================
# DATA PREPARATION FOR NIXTLA
# ==========================================
df_uni = df_uni.rename(columns={'load': 'y'})
if 'unique_id' not in df_uni.columns:
    df_uni['unique_id'] = 'DK1_Grid'
if 'ds' not in df_uni.columns:
    df_uni['ds'] = df_uni.index

df_uni['ds'] = pd.to_datetime(df_uni['ds'], utc=True).dt.tz_convert(None)

fut_exog_vars = [
    'tod_sin', 'tod_cos', 'dow_sin', 'dow_cos', 'doy_sin', 'doy_cos', 'DT1', 'DT2', 'DT3',
    'Total_Wind', 'Solar_Log', 't2m',
]

strict_cols = ['unique_id', 'ds', 'y'] + fut_exog_vars
df_uni = df_uni[strict_cols].copy()

for col in fut_exog_vars + ['y']:
    df_uni[col] = pd.to_numeric(df_uni[col], errors='coerce').astype('float32')

df_uni = df_uni.dropna().sort_values(['unique_id', 'ds']).reset_index(drop=True)
print(f"Data Pipeline Cleared. Shape: {df_uni.shape}")

1. Initiating Sterile Data Pipeline & Feature Engineering...
Data Pipeline Cleared. Shape: (17544, 15)


In [ ]:
# Calendar
fut_exog_vars = [
    'tod_sin', 'tod_cos', 'dow_sin', 'dow_cos', 'doy_sin', 'doy_cos', 'DT1', 'DT2', 'DT3',
]

In [ ]:
# Meteorological & Generation
fut_exog_vars = [
    'Total_Wind', 'Solar_Log', 't2m',
]

In [12]:
from sklearn.metrics import mean_squared_error
from neuralforecast.losses.pytorch import MSE
from sklearn.metrics import mean_absolute_percentage_error

In [ ]:
# ==========================================
# TEMPORAL TRAIN/VAL SPLIT
# ==========================================
HORIZON = 36
val_days = 30
val_size = val_days * 24

last_date = df_uni['ds'].dt.date.max()

# Defining 23:00 cutoff to force an 11:00 AM cutoff internally
dam_cutoff_end = pd.to_datetime(f"{last_date} 23:00:00")
df_full_aligned = df_uni[df_uni['ds'] <= dam_cutoff_end].copy()

train_df = df_full_aligned.iloc[:-val_size].copy()
val_df = df_full_aligned.iloc[-val_size:].copy()

max_lookback = 168
inference_df = df_full_aligned.iloc[-(val_size + max_lookback):].copy()

# ==========================================
# OPTUNA OBJECTIVE
# ==========================================
def objective(trial):
    input_size = trial.suggest_categorical('input_size', [168])
    hidden_size = trial.suggest_int('hidden_size', 64, 512, step=64)
    temporal_ff = trial.suggest_int('temporal_ff', 128, 512, step=64)
    channel_ff = trial.suggest_int('channel_ff', 8, 24, step=4)
    batch_size = trial.suggest_categorical('batch_size', [16, 32, 64, 128, 256])
    temporal_dropout = trial.suggest_float('temporal_dropout', 0.0, 0.4, step=0.1)
    learning_rate = trial.suggest_float('learning_rate', 1e-5, 1e-3, log=True)

    model = XLinear(
        h=HORIZON,
        n_series=1,
        #futr_exog_list=fut_exog_vars,
        input_size=input_size,
        hidden_size=hidden_size,
        temporal_ff=temporal_ff,
        channel_ff=channel_ff,
        temporal_dropout=temporal_dropout,
        loss=MSE(),
        learning_rate=learning_rate,
        batch_size=batch_size,
        max_steps=1000,
        scaler_type='standard',
        random_seed=42
    )

    nf = NeuralForecast(models=[model], freq='H')

    cv_df = nf.cross_validation(
        df=df_full_aligned,
        n_windows=7,
        step_size=24
    )
    cv_df['market_date'] = (cv_df['cutoff'] + pd.Timedelta(days=1)).dt.date
    market_cv_final = cv_df[cv_df['ds'].dt.date == cv_df['market_date']].copy()
    # Calculate MAPE
    mape = mean_absolute_percentage_error(market_cv_final['y'], market_cv_final['XLinear']) * 100



    return mape



# ==========================================
# RUNNING THE STUDY
# ==========================================
print("\n2. Initiating Optuna Study...")


study = optuna.create_study(
    direction='minimize',
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=5)
)

study.optimize(objective, n_trials=30, show_progress_bar=True)

print("\n=========================================================")
print(f"Optimization Finished!")
print(f"Best Trial MAPE: {study.best_value:.3f}%")
print(f"Best Parameters:")
for key, value in study.best_params.items():
    print(f"    {key}: {value}")
print("=========================================================")

[I 2026-05-22 12:39:47,389] A new study created in memory with name: no-name-fb70699a-16fc-4b1c-9256-5a92e0914eee



2. Initiating Optuna Study...


  0%|          | 0/30 [00:00<?, ?it/s]Seed set to 42
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs

  | Name            | Type          | Params | Mode 
----------------------------------------------------------
0 | loss            | MSE           | 0      | train
1 | padder_train    | ConstantPad1d | 0      | train
2 | scaler          | TemporalNorm  | 0      | train
3 | projection      | Sequential    | 32.4 K | train
4 | temporal_gating | GatingBlock   | 295 K  | train
5 | channel_gating  | GatingBlock   | 62     | train
6 | head            | Sequential    | 13.9 K | train
  | other params    | n/a           | 192    | n/a  
----------------------------------------------------------
342 K     Trainable params
0         Non-trainable params
342 K     Total params
1.369     Total estimated model params size (MB)
23        Modules in train mode
0         Modules in eval mode


c:\Users\scben\Documents\Obsidian\Benedek\Project-Thesis\BEN\.venv\Lib\site-packages\pytorch_lightning\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Epoch 999: 100%|██████████| 1/1 [00:00<00:00, 26.31it/s, v_num=210, train_loss_step=0.264, train_loss_epoch=0.264]

...

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
c:\Users\scben\Documents\Obsidian\Benedek\Project-Thesis\BEN\.venv\Lib\site-packages\pytorch_lightning\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Predicting DataLoader 0: 100%|██████████| 1/1 [00:00<00:00, 100.06it/s]


[I 2026-05-22 12:53:09,261] Trial 29 finished with value: 5.698971450328827 and parameters: {'input_size': 168, 'hidden_size': 256, 'temporal_ff': 384, 'channel_ff': 20, 'batch_size': 128, 'temporal_dropout': 0.4, 'learning_rate': 0.00033010551299655195}. Best is trial 12 with value: 4.725424572825432.


Best trial: 12. Best value: 4.72542: 100%|██████████| 30/30 [13:21<00:00, 26.73s/it]


Optimization Finished!
Best Trial MAPE: 4.725%
Best Parameters:
    input_size: 168
    hidden_size: 384
    temporal_ff: 128
    channel_ff: 20
    batch_size: 16
    temporal_dropout: 0.1
    learning_rate: 0.0008771518699386982


In [14]:
import pandas as pd

print("\n--- Top 10 Best Optuna Trials ---")

# 1. Convert the entire study to a DataFrame
trials_df = study.trials_dataframe()

# 2. Filter strictly for completed trials (ignores pruned/failed ones)
completed_trials = trials_df[trials_df['state'] == 'COMPLETE']

# 3. Sort by the objective value (MSE is minimized, so ascending=True)
top_10_trials = completed_trials.sort_values(by='value', ascending=True).head(10)

# 4. Clean up the columns for a readable output
# Optuna prefixes hyperparameters with 'params_'
param_cols = [col for col in top_10_trials.columns if col.startswith('params_')]
display_cols = ['number', 'value'] + param_cols

# Rename the 'value' column to represent your actual metric for clarity
clean_df = top_10_trials[display_cols].rename(columns={'value': 'MSE', 'number': 'Trial'})

# 5. Print the formatted table
print(clean_df.to_string(index=False))


--- Top 10 Best Optuna Trials ---
 Trial      MSE  params_batch_size  params_channel_ff  params_hidden_size  params_input_size  params_learning_rate  params_temporal_dropout  params_temporal_ff
    12 4.725425                 16                 20                 384                168              0.000877                      0.1                 128
    13 4.762060                 16                 20                 384                168              0.000910                      0.1                 128
    25 4.889862                256                 20                 512                168              0.000671                      0.1                 128
    11 4.958840                 16                 16                 512                168              0.000951                      0.1                 128
    10 4.981332                 16                 16                 512                168              0.000992                      0.1                 128
    2